# Getting Started with Strands Agents

This notebook introduces the core concepts of [Strands Agents](https://strandsagents.com) that you will use throughout the workshop:

1. Creating an agent
2. Choosing a model provider
3. Building tools with `@tool`
4. Using lifecycle hooks
5. Multi-agent swarms

**If you are already familiar with Strands Agents, skip this notebook and go to `01-graphrag-demo/`.**

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install strands-agents boto3 python-dotenv

print("✅ Environment ready")

---
## 1. Creating an Agent

A Strands agent combines a large language model (LLM) with tools. The agent uses the LLM to reason about the user's request and decide which tools to call.

By default, Strands uses **Amazon Bedrock** as the model provider — no extra configuration needed if your AWS credentials are set.

In [ ]:
from strands import Agent

# Create an agent with a system prompt.
# The system prompt defines the agent's personality and instructions.
agent = Agent(
    system_prompt="You are a helpful travel assistant. Answer questions about hotels and travel.",
)

# Talk to the agent — it uses Amazon Bedrock Claude by default
response = agent("What should I consider when booking a hotel in Lisbon?")
print(response)

---
## 2. Model Providers

Strands supports multiple model providers. This workshop uses Amazon Bedrock, but you can switch to others.

See all providers: [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/)

In [ ]:
from strands import Agent

# Option 1: Amazon Bedrock (default — no import needed)
agent_bedrock = Agent(system_prompt="You are a helpful assistant.")

# Option 2: Specify a Bedrock model explicitly
agent_specific = Agent(
    model="us.anthropic.claude-sonnet-4-20250514-v1:0",
    system_prompt="You are a helpful assistant.",
)

# Test it
response = agent_specific("Say hello in one sentence.")
print(response)

---
## 3. Creating Tools

Tools are Python functions decorated with `@tool`. The agent reads the function name and docstring to decide when to call each tool.

**Docstrings are critical** — the agent uses them to match user queries to tools. A vague docstring leads to wrong tool selection (Module 2 demonstrates this).

See: [Strands Tools Documentation](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/)

In [ ]:
from strands import Agent, tool

@tool
def search_hotels(city: str, max_price: int = 500) -> str:
    """Search for available hotels in a city under a maximum price per night."""
    # In a real application, this would query a database.
    # For this demo, we return a simulated response.
    return f"Found 3 hotels in {city} under ${max_price}/night:\n" \
           f"  1. AnyCompany {city} Resort ($95/night)\n" \
           f"  2. AnyCompany {city} Central ($110/night)\n" \
           f"  3. AnyCompany {city} Budget ($65/night)"

@tool
def book_hotel(hotel_name: str, guest_name: str, nights: int = 1) -> str:
    """Book a hotel room for a guest. Returns a booking confirmation ID."""
    total = nights * 95  # Simulated price
    return f"Booking confirmed: {guest_name} at {hotel_name}\n" \
           f"  {nights} night(s), total: ${total}\n" \
           f"  Confirmation ID: BK-001"

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 22°C, sunny"

# Create an agent with tools
agent = Agent(
    tools=[search_hotels, book_hotel, get_weather],
    system_prompt="You are a hotel booking assistant.",
)

print("Agent created with 3 tools:", [t.__name__ for t in [search_hotels, book_hotel, get_weather]])

In [ ]:
# The agent decides which tool to call based on the query
print("=== Test 1: Hotel search ===")
agent("Find me hotels in Lisbon under $100")

# You should see the agent call search_hotels and return the results

In [ ]:
print("=== Test 2: Weather (different tool) ===")
agent("What's the weather in Paris?")

# The agent should call get_weather, NOT search_hotels

In [ ]:
print("=== Test 3: Booking ===")
agent("Book AnyCompany Lisbon Resort for Alice for 3 nights")

# The agent should call book_hotel with the correct parameters

---
## 4. Lifecycle Hooks

Hooks intercept the agent's execution at specific points. The most important hook for this workshop is `BeforeToolCallEvent` — it fires after the LLM decides to call a tool but **before** the tool executes.

Setting `event.cancel_tool` prevents the tool from executing. The LLM receives the cancellation message instead of the tool result. **The LLM cannot bypass this** — it happens at the framework level.

See: [Strands Hooks Documentation](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/)

In [ ]:
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent

@tool
def book_room(hotel: str, guests: int = 1) -> str:
    """Book a hotel room for a number of guests."""
    return f"SUCCESS: Booked {hotel} for {guests} guests"

class MaxGuestsHook(HookProvider):
    """Block bookings with more than 10 guests."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check)

    def check(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] == "book_room":
            guests = event.tool_use["input"].get("guests", 1)
            if guests > 10:
                # This BLOCKS the tool call — the LLM cannot override it
                event.cancel_tool = f"BLOCKED: {guests} guests exceeds maximum of 10"

# Agent WITHOUT hook — no protection
agent_no_hook = Agent(tools=[book_room], system_prompt="You are a booking assistant.")

# Agent WITH hook — enforces the rule
agent_with_hook = Agent(tools=[book_room], hooks=[MaxGuestsHook()], system_prompt="You are a booking assistant.")

print("Created two agents: one without hook, one with MaxGuestsHook")

In [ ]:
print("=== Without hook: books 15 guests (no protection) ===")
agent_no_hook("Book AnyCompany Lisbon Resort for 15 guests")

# The agent books 15 guests — no validation

In [ ]:
print("=== With hook: BLOCKS 15 guests ===")
agent_with_hook("Book AnyCompany Lisbon Resort for 15 guests")

# The hook blocks the call — agent reports the error to the user

---
## 5. Multi-Agent Swarms

A swarm is a group of agents that hand off work to each other. Each agent has a specific role. Strands manages the handoff chain automatically.

See: [Strands Multi-Agent Documentation](https://strandsagents.com/docs/user-guide/concepts/multi-agent/)

In [ ]:
from strands import Agent, tool
from strands.multiagent import Swarm

@tool
def lookup_hotel(name: str) -> str:
    """Look up hotel details by name."""
    hotels = {"AnyCompany Lisbon Resort": "4 stars, $95/night, pool, spa"}
    return hotels.get(name, f"Hotel '{name}' not found in database")

# Three agents with different roles
executor = Agent(
    name="Executor",
    tools=[lookup_hotel],
    system_prompt="You look up hotel information using your tools. Report exactly what the tool returns.",
)

validator = Agent(
    name="Validator",
    system_prompt="You verify if the Executor's response is consistent. Check for fabricated data. Hand off to Critic.",
)

critic = Agent(
    name="Critic",
    system_prompt="You give a final verdict: VALID or SUSPICIOUS. Be brief.",
)

swarm = Swarm(nodes=[executor, validator, critic], max_handoffs=5)

print("Swarm created: Executor → Validator → Critic")

In [ ]:
print("=== Valid query ===")
swarm("What are the details for AnyCompany Lisbon Resort?")

# Executor looks up the hotel, Validator checks, Critic approves

In [ ]:
print("=== Invalid query (hotel does not exist) ===")
swarm("What are the details for AnyCompany Antarctica Lodge?")

# Executor gets 'not found', Validator flags it, Critic marks SUSPICIOUS

---
## Summary

| Concept | What it does | Workshop Module |
|---------|-------------|----------------|
| `Agent` + `@tool` | LLM reasons and calls functions | All modules |
| Model providers | Choose Bedrock, Anthropic, OpenAI, etc. | All modules |
| `BeforeToolCallEvent` + `cancel_tool` | Block tool calls that violate rules | Modules 4, 5, 6 |
| `Swarm` | Multi-agent handoff chain | Module 3 |
| `tool_registry` + `swap_tools` | Change tools at runtime | Module 2 |
| `MCPClient` | Connect to external tool servers | Module 6 |

You are ready to start the workshop. Go to `01-graphrag-demo/` for Module 1.